In [1]:
# Cell 1 — Đọc raw
import pm4py
import pandas as pd

log = pm4py.read_xes('../data/raw/BPI_2017.xes')
df_raw = pm4py.convert_to_dataframe(log)

print(f"Tổng số event: {len(df_raw)}")
print(f"Tổng số cột: {len(df_raw.columns)}")
print(f"\nDanh sách tất cả cột:")
for col in df_raw.columns:
    print(f"  {col}")

parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

Tổng số event: 1202267
Tổng số cột: 19

Danh sách tất cả cột:
  Action
  org:resource
  concept:name
  EventOrigin
  EventID
  lifecycle:transition
  time:timestamp
  case:LoanGoal
  case:ApplicationType
  case:concept:name
  case:RequestedAmount
  FirstWithdrawalAmount
  NumberOfTerms
  Accepted
  MonthlyCost
  Selected
  CreditScore
  OfferedAmount
  OfferID


In [2]:
# Cell 2 — Xem kiểu dữ liệu và tỉ lệ null từng cột
print("Kiểu dữ liệu và % null:\n")
for col in df_raw.columns:
    null_pct = df_raw[col].isna().mean() * 100
    dtype    = df_raw[col].dtype
    print(f"  {col:<45} {str(dtype):<15} null: {null_pct:.1f}%")

Kiểu dữ liệu và % null:

  Action                                        str             null: 0.0%
  org:resource                                  str             null: 0.0%
  concept:name                                  str             null: 0.0%
  EventOrigin                                   str             null: 0.0%
  EventID                                       str             null: 0.0%
  lifecycle:transition                          str             null: 0.0%
  time:timestamp                                datetime64[us, UTC] null: 0.0%
  case:LoanGoal                                 str             null: 0.0%
  case:ApplicationType                          str             null: 0.0%
  case:concept:name                             str             null: 0.0%
  case:RequestedAmount                          float64         null: 0.0%
  FirstWithdrawalAmount                         float64         null: 96.4%
  NumberOfTerms                                 float64         null: 

In [3]:
# Cell 3 — Xem 5 dòng đầu để hiểu format thực tế
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
df_raw.head(5)

,Action,org:resource,concept:name,EventOrigin,EventID,lifecycle:transition,time:timestamp,case:LoanGoal,case:ApplicationType,case:concept:name,case:RequestedAmount,FirstWithdrawalAmount,NumberOfTerms,Accepted,MonthlyCost,Selected,CreditScore,OfferedAmount,OfferID
0,Created,User_1,A_Create Application,Application,Application_652823628,complete,2016-01-01 09:51:15.304000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,statechange,User_1,A_Submitted,Application,ApplState_1582051990,complete,2016-01-01 09:51:15.352000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Created,User_1,W_Handle leads,Workflow,Workitem_1298499574,schedule,2016-01-01 09:51:15.774000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Deleted,User_1,W_Handle leads,Workflow,Workitem_1673366067,withdraw,2016-01-01 09:52:36.392000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Created,User_1,W_Complete application,Workflow,Workitem_1493664571,schedule,2016-01-01 09:52:36.403000+00:00,Existing loan takeover,New credit,Application_652823628,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Cell 4 — Khảo sát lifecycle 
if 'lifecycle:transition' in df_raw.columns:
    print("Các giá trị lifecycle:transition:")
    print(df_raw['lifecycle:transition'].value_counts())
    print(f"\nTổng event START:    {(df_raw['lifecycle:transition'].str.upper()=='START').sum()}")
    print(f"Tổng event COMPLETE: {(df_raw['lifecycle:transition'].str.upper()=='COMPLETE').sum()}")

Các giá trị lifecycle:transition:
lifecycle:transition
complete     475306
suspend      215402
schedule     149104
start        128227
resume       127160
ate_abort     85224
withdraw      21844
Name: count, dtype: int64

Tổng event START:    128227
Tổng event COMPLETE: 475306


In [5]:
# Cell 5 — Khảo sát case-level attributes
case_cols = [c for c in df_raw.columns if c.startswith('case:')]
print(f"Có {len(case_cols)} case-level attributes:\n")
for col in case_cols:
    unique_vals = df_raw[col].nunique()
    sample_vals = df_raw[col].dropna().unique()[:5]
    print(f"  {col}")
    print(f"    unique: {unique_vals}, sample: {sample_vals}\n")

Có 4 case-level attributes:

  case:LoanGoal
    unique: 14, sample: <ArrowStringArray>
['Existing loan takeover',       'Home improvement',                    'Car',
 'Other, see explanation',    'Remaining debt home']
Length: 5, dtype: str

  case:ApplicationType
    unique: 2, sample: <ArrowStringArray>
['New credit', 'Limit raise']
Length: 2, dtype: str

  case:concept:name
    unique: 31509, sample: <ArrowStringArray>
[ 'Application_652823628', 'Application_1691306052',  'Application_428409768',
 'Application_1746793196',  'Application_828200680']
Length: 5, dtype: str

  case:RequestedAmount
    unique: 701, sample: [20000. 10000. 15000.  5000. 35000.]



In [6]:
# Cell 6 — Khảo sát event-level attributes
event_cols = [c for c in df_raw.columns if not c.startswith('case:')]
print(f"Có {len(event_cols)} event-level attributes:\n")
for col in event_cols:
    unique_vals = df_raw[col].nunique()
    sample_vals = df_raw[col].dropna().unique()[:5]
    print(f"  {col}")
    print(f"    unique: {unique_vals}, sample: {sample_vals}\n")

Có 15 event-level attributes:

  Action
    unique: 5, sample: <ArrowStringArray>
['Created', 'statechange', 'Deleted', 'Obtained', 'Released']
Length: 5, dtype: str

  org:resource
    unique: 149, sample: <ArrowStringArray>
['User_1', 'User_17', 'User_52', 'User_11', 'User_117']
Length: 5, dtype: str

  concept:name
    unique: 26, sample: <ArrowStringArray>
[  'A_Create Application',            'A_Submitted',         'W_Handle leads',
 'W_Complete application',              'A_Concept']
Length: 5, dtype: str

  EventOrigin
    unique: 3, sample: <ArrowStringArray>
['Application', 'Workflow', 'Offer']
Length: 3, dtype: str

  EventID
    unique: 1202267, sample: <ArrowStringArray>
['Application_652823628',  'ApplState_1582051990',   'Workitem_1298499574',
   'Workitem_1673366067',   'Workitem_1493664571']
Length: 5, dtype: str

  lifecycle:transition
    unique: 7, sample: <ArrowStringArray>
['complete', 'schedule', 'withdraw', 'start', 'suspend']
Length: 5, dtype: str

  time:timest

In [ ]:
# Cell 7 — Xem 1 case cụ thể -> hiểu cấu trúc thực tế
first_case = df_raw['case:concept:name'].iloc[0]
case_detail = df_raw[df_raw['case:concept:name'] == first_case].sort_values('time:timestamp')
print(f"Chi tiết case: {first_case}")
print(f"Số event: {len(case_detail)}\n")
case_detail[['time:timestamp', 'concept:name', 'lifecycle:transition', 'org:resource']].to_string(index=False)

Chi tiết case: Application_652823628
Số event: 40



'                  time:timestamp             concept:name lifecycle:transition org:resource\n2016-01-01 09:51:15.304000+00:00     A_Create Application             complete       User_1\n2016-01-01 09:51:15.352000+00:00              A_Submitted             complete       User_1\n2016-01-01 09:51:15.774000+00:00           W_Handle leads             schedule       User_1\n2016-01-01 09:52:36.392000+00:00           W_Handle leads             withdraw       User_1\n2016-01-01 09:52:36.403000+00:00   W_Complete application             schedule       User_1\n2016-01-01 09:52:36.413000+00:00                A_Concept             complete       User_1\n2016-01-02 10:45:22.429000+00:00   W_Complete application                start      User_17\n2016-01-02 10:49:28.816000+00:00   W_Complete application              suspend      User_17\n2016-01-02 11:23:04.299000+00:00               A_Accepted             complete      User_52\n2016-01-02 11:29:03.994000+00:00           O_Create Offer            

In [ ]:
import json
import pandas as pd

# 1. Số sequence (Path đúng: data/raw/sop/regulation_graph.json)
with open('../data/raw/sop/regulation_graph.json') as f:
    reg = json.load(f)
print('Activities :', len(reg['activities']))
print('Sequences  :', len(reg['sequences']))    # đáp án: 7
print('Conditions :', len(reg['conditions']))
print('Roles      :', len(reg['roles']))

# 2. Phạm vi thời gian dataset
df = pd.read_parquet('../data/processed/event_log_clean.parquet')
print('\nThời gian:')
print('  Min:', df['timestamp'].min())
print('  Max:', df['timestamp'].max())
print('  Số case:', df['case_id'].nunique())
print('  Số event:', len(df))

# 3. Số case happy-path (đáp án: 24895, không phải 18504)
df_app = df[df['event_origin'] == 'Application'].copy()
df_app = df_app.sort_values(['case_id', 'seq_index'])
end_events = df_app.groupby('case_id').last()
happy_set = {'A_Pending', 'A_Cancelled', 'A_Complete'}
happy_cases = end_events[end_events['activity'].isin(happy_set)].index
print(f'\nHappy-path cases: {len(happy_cases)}')

# 4. Số quan hệ trong RKG (cần Neo4j)
import os, sys
sys.path.append('../')
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    queries = [
        ('MUST_PRECEDE',  "MATCH ()-[r:MUST_PRECEDE]->() RETURN count(r) AS cnt"),
        ('REQUIRES',      "MATCH ()-[r:REQUIRES]->() RETURN count(r) AS cnt"),
        ('PERFORMED_BY',  "MATCH ()-[r:PERFORMED_BY]->() RETURN count(r) AS cnt"),
        ('DEFINED_IN',    "MATCH ()-[r:DEFINED_IN]->() RETURN count(r) AS cnt"),
    ]
    print('\nQuan hệ trong Neo4j:')
    for name, q in queries:
        result = session.run(q).single()
        print(f'  {name:<14}: {result["cnt"]}')

driver.close()

Activities : 13
Sequences  : 6
Conditions : 4
Roles      : 3

Thời gian:
  Min: 2016-01-01 09:51:15.304000+00:00
  Max: 2017-02-01 14:11:03.499000+00:00
  Số case: 28512
  Số event: 1047482

Happy-path cases: 24895

Quan hệ trong Neo4j:
  MUST_PRECEDE  : 6
  REQUIRES      : 0
  PERFORMED_BY  : 13
  DEFINED_IN    : 17


: 